# Artefato 2: Modelo Clássico Baseline (Machine Learning Tradicional)


### Baseline de Regressão Logistica

Treinar RL com regularização L2 e validar em conjunto de validação; registrar métricas. Entrega: função e resultados.

In [12]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import json
import re
import sys
import os

In [13]:
df_pixels = pd.read_csv('pixels_dataset.csv')

In [14]:
df_pixels.head()

,path,filename,count,height,width,dtype,crs,transform,pixel_0,pixel_1,...,pixel_147446,pixel_147447,pixel_147448,pixel_147449,pixel_147450,pixel_147451,pixel_147452,pixel_147453,pixel_147454,pixel_147455
0,/content/drive/Shareddrives/Banco de Imagens M...,chip_128x128_multiband.tif,9,128,128,int16,EPSG:4326,"(0.00013871468102456285, 0.0, -46.612019676870...",44,44,...,86,86,86,86,84,84,86,84,84,84
1,/content/drive/Shareddrives/Banco de Imagens M...,chip_128x128_multiband.tif,9,128,128,int16,EPSG:4326,"(0.00013871468102456285, 0.0, -46.623394280714...",52,50,...,92,92,96,96,96,96,96,96,96,96
2,/content/drive/Shareddrives/Banco de Imagens M...,chip_128x128_multiband.tif,9,128,128,int16,EPSG:4326,"(0.00013871468102456285, 0.0, -46.612574535594...",52,55,...,84,88,88,86,86,82,82,76,76,84
3,/content/drive/Shareddrives/Banco de Imagens M...,chip_128x128_multiband.tif,9,128,128,int16,EPSG:4326,"(0.00013871468102456285, 0.0, -46.535865316988...",50,49,...,75,73,73,71,71,76,76,86,86,86
4,/content/drive/Shareddrives/Banco de Imagens M...,chip_128x128_multiband.tif,9,128,128,int16,EPSG:4326,"(0.00013871468102456285, 0.0, -46.635046313920...",55,55,...,92,92,96,96,96,96,98,98,96,96


In [16]:
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("__file__"), "../../src")))
from models.pipeline_factory import train_classification 

# --- CONSTANTES ADAPTADAS PARA SUA TASK ---
MODEL_NAME = "logisticregression"
MODEL_PARAMS = {
    "penalty": "l2",      
    "C": 1.0,               
    "solver": "lbfgs",     
    "max_iter": 1000,       
    "random_state": 42
}

PIXELS_DATASET_PATH = "pixels_dataset.csv"
EXTRACTED_CODES_PATH = "extracted_codes.json"

def prepare_dataset(dataset_path: str, extracted_codes_path: str):
    df = pd.read_csv(dataset_path)

    with open(extracted_codes_path, 'r') as f:
        extracted_codes = json.load(f)

    # Lógica de extração de ID do colega
    all_ids = extracted_codes['negativos'] + extracted_codes['positivos']
    all_ids_sorted = sorted(all_ids, key=len, reverse=True)
    id_pattern = '|'.join(re.escape(id_val) for id_val in all_ids_sorted)
    regex = rf'({id_pattern})'
    df['image_id'] = df['path'].str.extract(regex)[0]

    # Criação do Label (1=Positivo, 0=Negativo)
    df['label'] = df['image_id'].apply(
        lambda x: 1 if x in extracted_codes['positivos'] else (
            0 if x in extracted_codes['negativos'] else -1
        )
    )

    # Removemos os -1 (casos que não deram match no JSON) para não sujar a baseline
    df = df[df['label'] != -1].reset_index(drop=True)

    X = df.drop(
        columns=[
            'path', 'filename', 'count', 'height', 'width', 'dtype',
            'crs', 'transform', 'image_id', 'label'
        ]
    )
    y = df["label"]

    return X, y

if __name__ == "__main__":
    print(f"Iniciando Task: Baseline {MODEL_NAME}...")
    

    X, y = prepare_dataset(PIXELS_DATASET_PATH, EXTRACTED_CODES_PATH)
    print(f"Dataset preparado. Shape: {X.shape}. Classes: {np.bincount(y)}")

    pipeline, metrics = train_classification(
        X,
        y,
        model_name=MODEL_NAME,
        **MODEL_PARAMS
    )

    print("\n=== RESULTADOS BASELINE REGRESSÃO LOGÍSTICA (L2) ===")
    for k, v in metrics.items():
        print(f"{k.upper()}: {v:.4f}")

Iniciando Task: Baseline logisticregression...


/var/folders/st/r72pjbts3bb1zfh2s1r4pzw00000gn/T/ipykernel_50468/3126902967.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['image_id'] = df['path'].str.extract(regex)[0]
/var/folders/st/r72pjbts3bb1zfh2s1r4pzw00000gn/T/ipykernel_50468/3126902967.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['label'] = df['image_id'].apply(


Dataset preparado. Shape: (295, 147456). Classes: [179 116]


/Users/pedroauler/miniforge3/envs/Aster/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



=== RESULTADOS BASELINE REGRESSÃO LOGÍSTICA (L2) ===
ACCURACY: 0.7797
PRECISION: 0.6818
RECALL: 0.7143
F1: 0.6977
ROC_AUC: 0.8120
